In [1]:
## Bookharvest Webscrapping
#importing required libraries

import requests
from bs4 import BeautifulSoup
import pandas as pd
import psycopg2
from datetime import datetime
from dotenv import load_dotenv
import os
from urllib.parse import urljoin

In [2]:
### Extraction from Website (Scrape the Website)
#scraping using the web url

print('Establishing connection to the website for scrapping')

bookharvest_url = "https://books.toscrape.com/catalogue/page-1.html"
response = requests.get(bookharvest_url)

print(f"Status Code: {response.status_code}")

if response.status_code == 200:
    print("Website is reachable and ready to scrape. ")
else:
    print("Website not reachable. Check internet connection. ")

Establishing connection to the website for scrapping
Status Code: 200
Website is reachable and ready to scrape. 


In [3]:
#using BeautifulSoup to access the HTML

bookharvest_soup = BeautifulSoup(response.text, "html.parser")
bookharvest_cards = bookharvest_soup.find_all("article", class_="product_pod")

#checking the number id books extracted
print(f"This page contains {len (bookharvest_cards)} books")

#printing out tags and making them strings 
print(bookharvest_cards[0].prettify()[:600])

This page contains 20 books
<article class="product_pod">
 <div class="image_container">
  <a href="a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="../media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="p


In [4]:
# Dictionary creation, to change letters to numbers
rating_maps = {
    "One":1,
    "Two":2,
    "Three":3,
    "Four":4,
    "Five":5
}
#creating a list to loop through the entire 50 pages

bookharvest_pages = []

Total_pages = 50

print(f" Scraping {Total_pages} pages in progress... \n")

for page_number in range (1, Total_pages + 1):
    loop_url = f"https://books.toscrape.com/catalogue/page-{page_number}.html"

    loop_response = requests.get(loop_url)

    # creating a condition to skip a page when there is an error
    if loop_response.status_code != 200:
        print(f"Could not load page {page_number}, skipping...")
        continue

    loop_soup = BeautifulSoup(loop_response.text, "html.parser")
    loop_cards = loop_soup.find_all("article",class_="product_pod")

    #
    for cards in loop_cards:

    # extract title
        book_title = cards.find("img")["alt"]

        # extract price
        book_pricetext = cards.find("p", class_="price_color").text
        book_price = float(book_pricetext.replace("£", "").replace("Â", "").strip())

        # extract rating
        rating_words = cards.find("p", class_="star-rating")["class"][1]
        book_rating = rating_maps.get(rating_words, 0)

        # extract availability
        book_availability = cards.find(
            "p", class_="instock availability"
        ).text.strip()

        # extract book link
        book_link = cards.find("h3").find("a")["href"]

        # convert relative link into full URL
        book_url = urljoin(loop_url, book_link)

        # visit individual book page
        book_response = requests.get(book_url)

        # extract genre
        book_soup = BeautifulSoup(book_response.text, "html.parser")

        breadcrumb = book_soup.find("ul", class_="breadcrumb")
        breadcrumb_items = breadcrumb.find_all("li")

        book_genre = breadcrumb_items[2].get_text(strip=True)

        # save everything
        bookharvest_pages.append({
            "title": book_title,
            "price_gdp": book_price,
            "rating": book_rating,
            "availability": book_availability,
            "genre": book_genre
        })
        #print looping progress
        if page_number % 10 ==0 or page_number ==1:
            print(f"page {page_number} done : {len(bookharvest_pages)} collected so far")

print("looping completed")


 Scraping 50 pages in progress... 

page 1 done : 1 collected so far
page 1 done : 2 collected so far
page 1 done : 3 collected so far
page 1 done : 4 collected so far
page 1 done : 5 collected so far
page 1 done : 6 collected so far
page 1 done : 7 collected so far
page 1 done : 8 collected so far
page 1 done : 9 collected so far
page 1 done : 10 collected so far
page 1 done : 11 collected so far
page 1 done : 12 collected so far
page 1 done : 13 collected so far
page 1 done : 14 collected so far
page 1 done : 15 collected so far
page 1 done : 16 collected so far
page 1 done : 17 collected so far
page 1 done : 18 collected so far
page 1 done : 19 collected so far
page 1 done : 20 collected so far
page 10 done : 181 collected so far
page 10 done : 182 collected so far
page 10 done : 183 collected so far
page 10 done : 184 collected so far
page 10 done : 185 collected so far
page 10 done : 186 collected so far
page 10 done : 187 collected so far
page 10 done : 188 collected so far
page 

In [5]:
#converting to a dataframe
df = pd.DataFrame(bookharvest_pages)

df.head()

,title,price_gdp,rating,availability,genre
0,A Light in the Attic,51.77,3,In stock,Poetry
1,Tipping the Velvet,53.74,1,In stock,Historical Fiction
2,Soumission,50.10,1,In stock,Fiction
3,Sharp Objects,47.82,4,In stock,Mystery
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,History


In [6]:
from dotenv import load_dotenv
import os
import psycopg2

load_dotenv()

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
)

cursor = conn.cursor()

print("Connected successfully!")

Connected successfully!


In [10]:
from urllib.parse import quote_plus
from sqlalchemy import create_engine
import os

load_dotenv()

password = quote_plus(os.getenv("DB_PASSWORD"))

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{password}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)


In [14]:
df.to_sql(
    "bookharvest",
    engine,
    if_exists="append",
    index=False
)

1000